# Day 5: Session 5A - The Split-Apply-Combine Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/5a_grouping_data.html)

Date: 09/04/2026

In [1]:
import pandas as pd

url = "https://eds-217-essential-python.github.io/data/messy_field_survey.csv"
survey = pd.read_csv(url)

survey = survey.drop_duplicates()

survey["site"] = survey["site"].str.strip().str.lower().str.replace("-", "_")
# survey['site'] = survey['site'].str.lower()
# survey['site'] = survey['site'].str.replace('-', '_')

survey["pH"] = survey["pH"].str.replace(",", ".").astype(float)
# survey['pH'] = survey['pH'].astype(float)

survey = survey.dropna(
    subset=["temperature_c", "dissolved_oxygen_mg_L", "conductivity_uS_cm"]
)
survey["n_replicates"] = survey["n_replicates"].fillna(1).astype(int)
#survey["n_replicates"] = survey["n_replicates"].astype(int)

survey = survey[survey["temperature_c"] > -100].copy()
survey = survey.rename(columns={"collection date": "collection_date"})

survey.shape

(255, 7)

### .groupby()
Splits the table into one group per site, applies the same calculation to each group, and combines the answers into a single result, labelled by group.

In [2]:
grouped = survey.groupby('site')
grouped
# this is just the split step

In [3]:
result = grouped['dissolved_oxygen_mg_L'].mean()
# this step picks a column and names the calculation

In [4]:
result['site_a']

9.485909090909091

In [5]:
survey.groupby('site')['dissolved_oxygen_mg_L'].mean()

# this is the same pattern but on one line

# general pattern:
# dataframe.groupby('key')['column'].aggregation()
#                    split      pick        apply

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64

Reading the result

In [6]:
mean_do = survey.groupby('site')['dissolved_oxygen_mg_L'].mean()

print(type(mean_do))
print(mean_do.index)

# the result is a series with an index made up of the group labels
# i.e. the site labels are now labels of rows

<class 'pandas.core.series.Series'>
Index(['site_a', 'site_b', 'site_c', 'site_d', 'site_e', 'site_f'], dtype='object', name='site')


In [7]:
print(mean_do['site_c'])
print(mean_do.idxmax())
print('Best site:', mean_do.idxmax())
print('Worst site:', mean_do.idxmin())

6.873111111111111
site_d
Best site: site_d
Worst site: site_f


In [8]:
# Group survey by site and take the mean of temperature_c, 
# then use .idxmax() on the result to name the warmest site. 
# Compare it with the worst site for dissolved oxygen above. 
# Are they the same site?

temp_grouped = survey.groupby('site')['temperature_c'].mean()

temp_grouped.idxmax()

'site_f'

In [9]:
survey.groupby('site')['dissolved_oxygen_mg_L'].max()

site
site_a    11.12
site_b     9.86
site_c     7.99
site_d    11.43
site_e     8.79
site_f     7.77
Name: dissolved_oxygen_mg_L, dtype: float64

In [10]:
survey.groupby('site')['dissolved_oxygen_mg_L'].min()

site
site_a    8.21
site_b    7.33
site_c    5.65
site_d    8.60
site_e    5.16
site_f    4.14
Name: dissolved_oxygen_mg_L, dtype: float64

In [11]:
survey.groupby('site')['n_replicates'].sum()

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

In [12]:
survey.groupby('site')['dissolved_oxygen_mg_L'].count()

site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64

In [13]:
# What is the highest pH recorded at each site?
# How many bottles (n_replicates) were filled in total at each site? 

survey.groupby('site')['pH'].max()

site
site_a    7.92
site_b    7.41
site_c    7.10
site_d    8.33
site_e    7.47
site_f    6.98
Name: pH, dtype: float64

In [14]:
survey.groupby('site')['n_replicates'].sum()

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

In [15]:
def classify_ph(value):
    """Label a pH value as acidic, neutral, or alkaline."""
    if value < 6.5:
        return 'acidic'
    elif value > 7.5:
        return 'alkaline'
    else:
        return 'neutral'


survey['ph_class'] = survey['pH'].apply(classify_ph)
survey['ph_class'].value_counts()

ph_class
neutral     171
acidic       42
alkaline     42
Name: count, dtype: int64

In [16]:
survey.groupby('ph_class')['dissolved_oxygen_mg_L'].mean()

ph_class
acidic      6.350000
alkaline    9.948571
neutral     8.019181
Name: dissolved_oxygen_mg_L, dtype: float64

In [17]:
print(survey.groupby('site')['dissolved_oxygen_mg_L'].count())
print(survey.groupby('site')['dissolved_oxygen_mg_L'].mean())
print(survey.groupby('site')['temperature_c'].mean())

site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64
site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
site
site_a    16.600000
site_b    18.102439
site_c    21.911111
site_d    15.179487
site_e    19.730233
site_f    23.297674
Name: temperature_c, dtype: float64
